# SafeRoute Analytics — API-key-free accident maps

This notebook displays the accident **heatmap**, **cluster map**, and **individual points** from `data/accidents.csv` using Folium and OpenStreetMap tiles. No Google Maps, Mapbox, or other map API key is required.

> Map tiles are fetched from OpenStreetMap's public tile service, so an internet connection is needed to render the basemap. Accident data and map layers are generated locally from the CSV.


In [ ]:
%pip install -q pandas folium


In [ ]:
from pathlib import Path
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import display, HTML

# Works when this notebook is in the project root. If needed, change DATA_PATH.
DATA_PATH = Path("data") / "accidents.csv"
df = pd.read_csv(DATA_PATH)

required = {"latitude", "longitude"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

# Keep only valid geographic coordinates.
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"]).copy()
df = df[df["latitude"].between(-90, 90) & df["longitude"].between(-180, 180)]

print(f"Accident records available for mapping: {len(df):,}")
display(df.head())


## 1. Heatmap

The heatmap shows areas with higher concentrations of recorded accidents. The `HeatMap` layer is rendered by Folium in the notebook; no map API key is used.


In [ ]:
center = [float(df["latitude"].mean()), float(df["longitude"].mean())]
heatmap_map = folium.Map(location=center, zoom_start=5, tiles="OpenStreetMap", control_scale=True)

heat_data = df[["latitude", "longitude"]].values.tolist()
HeatMap(heat_data, radius=13, blur=18, min_opacity=0.25, name="Accident density").add_to(heatmap_map)
folium.LayerControl().add_to(heatmap_map)

display(heatmap_map)
heatmap_map.save("accident_heatmap.html")
print("Saved: accident_heatmap.html")


## 2. Cluster map

Nearby accident markers are grouped into interactive clusters. Zoom in or click a cluster to explore individual records.


In [ ]:
cluster_map = folium.Map(location=center, zoom_start=5, tiles="OpenStreetMap", control_scale=True)
marker_cluster = MarkerCluster(name="Accident clusters").add_to(cluster_map)

# Add popup details where the corresponding columns exist.
detail_cols = [c for c in ["accident_id", "city", "date", "severity", "cause", "fatalities", "injuries"] if c in df.columns]
for _, row in df.iterrows():
    details = "<br>".join(f"<b>{col.replace('_', ' ').title()}:</b> {row[col]}" for col in detail_cols)
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(details or "Road accident", max_width=320),
        tooltip=str(row.get("city", "Accident")),
    ).add_to(marker_cluster)

folium.LayerControl().add_to(cluster_map)
display(cluster_map)
cluster_map.save("accident_clusters.html")
print("Saved: accident_clusters.html")


## 3. Individual accident points

Each accident is displayed as a circle marker, coloured by severity when that column is available. Click a point to view its details.


In [ ]:
severity_colors = {
    "Fatal": "red",
    "Serious": "orange",
    "Minor": "green",
    "PDO": "blue",
}

points_map = folium.Map(location=center, zoom_start=5, tiles="OpenStreetMap", control_scale=True)
for _, row in df.iterrows():
    severity = str(row.get("severity", "Unknown"))
    details = "<br>".join(
        f"<b>{col.replace('_', ' ').title()}:</b> {row[col]}"
        for col in detail_cols
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=severity_colors.get(severity, "cadetblue"),
        fill=True,
        fill_opacity=0.75,
        weight=1,
        tooltip=f"{row.get('city', 'Accident')} — {severity}",
        popup=folium.Popup(details or "Road accident", max_width=320),
    ).add_to(points_map)

display(points_map)
points_map.save("accident_points.html")
print("Saved: accident_points.html")


## Notes

- No API key is needed for these Folium maps.
- Internet access is needed for the OpenStreetMap basemap tiles.
- The three HTML maps are saved in the notebook's current working directory and can be opened in a browser.
- For a very large dataset, consider sampling records in the individual-points view for better performance.
